# Cálculo Diferencial e Integral — caderno de consulta e prática

Material de estudo orientado ao uso de Python para calcular, verificar e interpretar resultados de Cálculo Diferencial e Integral.

## Como usar
- Execute as células na ordem, especialmente a seção de configuração.
- Use `SymPy` para cálculos simbólicos e `NumPy`/`Matplotlib` para aproximações e gráficos.
- Um resultado calculado não dispensa interpretar domínio, hipóteses e significado.
- Compare resultados simbólicos com aproximações numéricas ou gráficos quando isso ajudar a validar a interpretação.

**Instalação, se necessário:** `pip install sympy numpy matplotlib scipy`

## Índice
1. Configuração e funções
2. **Tema 2 — Limites, continuidade e assíntotas: resolução de exercícios**
   - escolha do bloco por tipo de exercício; funções por partes; continuidade; assíntotas; conferência
3. **Tema 3 — Derivadas: conceitos, propriedades e cálculos**
4. **Tema 4 — Derivadas: aplicações**
5. **Tema 5 — Integrais: conceitos, propriedades e técnicas de integração**
6. **Tema 6 — Integrais: aplicações**
7. Checklist de escolha da ferramenta e exercícios rápidos

## 1. Configuração

Execute esta seção uma vez antes de usar os exercícios. Ela importa as bibliotecas, define os símbolos e reúne as funções auxiliares utilizadas nos cálculos de limites, continuidade e assíntotas.

In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

x, h, a, C, t = sp.symbols("x h a C t", real=True)
e, pi, inf = sp.exp(1), sp.pi, sp.oo

# Catálogo de expressões geométricas; x representa a grandeza independente.
from types import SimpleNamespace

expressoes = SimpleNamespace(
    circulo=SimpleNamespace(area=sp.pi*x**2, comprimento=2*sp.pi*x),
    quadrado=SimpleNamespace(area=x**2, perimetro=4*x),
    esfera=SimpleNamespace(volume=sp.Rational(4,3)*sp.pi*x**3, area_superficial=4*sp.pi*x**2),
    cubo=SimpleNamespace(volume=x**3, area_superficial=6*x**2)
)

def calcular_taxa_relacionada(expressao, variavel, valor=None, taxa=None, funcao_tempo=None, instante=None):
    """Calcula a taxa temporal de uma expressão pela regra da cadeia."""
    variavel_tempo = sp.Function(str(variavel))(t)

    if funcao_tempo is not None:
        if instante is None:
            raise ValueError("Informe instante quando funcao_tempo estiver preenchida.")
        expressao_tempo = expressao.subs(variavel, funcao_tempo)
        derivada_tempo = sp.diff(expressao_tempo, t)
        valor_calculado = sp.simplify(funcao_tempo.subs(t, instante))
        taxa_calculada = sp.simplify(sp.diff(funcao_tempo, t).subs(t, instante))
        resultado = sp.simplify(derivada_tempo.subs(t, instante))
        return {"derivada_tempo": derivada_tempo, "valor": valor_calculado, "taxa": taxa_calculada, "resultado": resultado}

    if valor is None or taxa is None:
        raise ValueError("Informe valor e taxa, ou então funcao_tempo e instante.")
    expressao_tempo = expressao.subs(variavel, variavel_tempo)
    derivada_tempo = sp.diff(expressao_tempo, t)
    substituicoes = {variavel_tempo: valor, sp.diff(variavel_tempo, t): taxa}
    if instante is not None:
        substituicoes[t] = instante
    resultado = sp.simplify(derivada_tempo.subs(substituicoes))
    return {"derivada_tempo": derivada_tempo, "valor": valor, "taxa": taxa, "resultado": resultado}
def iguais(a, b):
    """Verifica se duas expressões simbólicas são equivalentes."""
    return sp.simplify(a - b) == 0

def formatar_limite_bilateral(esquerda, direita):
    """Devolve o limite comum ou informa que os limites laterais diferem."""
    return esquerda if iguais(esquerda, direita) else "não existe: os limites laterais são diferentes"

def valor_funcao(funcao, ponto):
    """Calcula o valor da função no ponto, quando definido."""
    try:
        valor = funcao.subs(x, ponto)
        return "não definido" if valor.has(sp.nan, sp.zoo) else valor
    except (AttributeError, TypeError, ValueError):
        return "não foi possível avaliar"

def ramo_lateral(funcao, ponto, lado):
    """Seleciona o ramo de Piecewise ativo imediatamente de um lado do ponto."""
    if not isinstance(funcao, sp.Piecewise):
        return funcao
    ponto = sp.sympify(ponto)
    escala = max(1.0, abs(float(sp.N(ponto))))
    amostra = ponto + (-sp.Float(escala*1e-7) if lado == "-" else sp.Float(escala*1e-7))
    for expressao, condicao in funcao.args:
        if condicao is True or condicao == sp.true:
            return expressao
        try:
            if condicao.subs(x, amostra) == sp.true:
                return expressao
        except (AttributeError, TypeError):
            pass
    raise ValueError(f"Não foi encontrado ramo ativo à {'esquerda' if lado == '-' else 'direita'} de x={ponto}.")

def limite_lateral(funcao, ponto, lado):
    """Calcula limite lateral, selecionando o ramo de Piecewise automaticamente."""
    return sp.limit(ramo_lateral(funcao, ponto, lado), x, ponto, dir=lado)

def ramo_no_infinito(funcao, sinal):
    """Seleciona o ramo ativo para x muito positivo ('+') ou negativo ('-')."""
    if not isinstance(funcao, sp.Piecewise):
        return funcao
    amostra = sp.Integer(10**8 if sinal == "+" else -(10**8))
    for expressao, condicao in funcao.args:
        if condicao is True or condicao == sp.true:
            return expressao
        try:
            if condicao.subs(x, amostra) == sp.true:
                return expressao
        except (AttributeError, TypeError):
            pass
    raise ValueError("Não foi encontrado ramo ativo nessa direção do infinito.")

def candidatos_transicao(funcao):
    """Extrai candidatos a transição das condições de Piecewise."""
    candidatos = set()
    if not isinstance(funcao, sp.Piecewise):
        return []
    for _, condicao in funcao.args:
        for relacao in condicao.atoms(sp.core.relational.Relational):
            equacao = sp.simplify(relacao.lhs-relacao.rhs)
            if equacao.has(x):
                try:
                    for solucao in sp.solve(sp.Eq(equacao, 0), x):
                        if solucao.is_real is not False and solucao.is_finite is not False:
                            candidatos.add(solucao)
                except (NotImplementedError, ValueError, TypeError):
                    pass
    return sorted(candidatos, key=lambda v: float(sp.N(v)))

def candidatos_denominador(funcao):
    """Encontra zeros reais candidatos dos denominadores dos ramos."""
    candidatos = set()
    expressoes = [expr for expr, _ in funcao.args] if isinstance(funcao, sp.Piecewise) else [funcao]
    for expr in expressoes:
        denominador = sp.factor(sp.denom(sp.together(expr)))
        if denominador == 1:
            continue
        try:
            for solucao in sp.solve(sp.Eq(denominador, 0), x):
                if solucao.is_real is not False and solucao.is_finite is not False:
                    candidatos.add(solucao)
        except (NotImplementedError, ValueError, TypeError):
            pass
    return sorted(candidatos, key=lambda v: float(sp.N(v)))

def analisar_assintotas(funcao):
    """Analisa assíntotas de funções racionais usuais e funções por partes."""
    verticais = []
    candidatos = sorted(set(candidatos_transicao(funcao)+candidatos_denominador(funcao)),
                        key=lambda v: float(sp.N(v)))
    for ponto in candidatos:
        try:
            le, ld = limite_lateral(funcao, ponto, "-"), limite_lateral(funcao, ponto, "+")
            if any(v in (inf, -inf) for v in (le, ld)):
                verticais.append({"x": ponto, "esquerda": le, "direita": ld})
        except (ValueError, TypeError, NotImplementedError):
            pass
    infinitas = []
    for sinal, direcao in (("+", inf), ("-", -inf)):
        try:
            ramo = ramo_no_infinito(funcao, sinal)
            limite = sp.limit(ramo, x, direcao)
            if limite.is_finite is True:
                infinitas.append({"direcao": direcao, "tipo": "horizontal", "reta": sp.Eq(sp.Symbol("y"), limite)})
                continue
            m = sp.limit(ramo/x, x, direcao)
            if m.is_finite is True and sp.simplify(m) != 0:
                b = sp.limit(ramo-m*x, x, direcao)
                if b.is_finite is True:
                    infinitas.append({"direcao": direcao, "tipo": "oblíqua", "reta": sp.Eq(sp.Symbol("y"), sp.expand(m*x+b))})
        except (ValueError, TypeError, NotImplementedError):
            pass
    return {"verticais": verticais, "infinitas": infinitas}

def mostrar_assintotas(resultado):
    """Imprime assíntotas identificadas."""
    print(f"ASSÍNTOTAS VERTICAIS")
    if resultado["verticais"]:
        for item in resultado["verticais"]:
            print(f"  x = {item['x']} (limite esquerdo: {item['esquerda']}; direito: {item['direita']})")
    else:
        print(f"  Nenhuma assíntota vertical identificada.")
    print(f"\nASSÍNTOTAS QUANDO x → ±∞")
    if resultado["infinitas"]:
        for item in resultado["infinitas"]:
            direcao = "+∞" if item["direcao"] == inf else "−∞"
            print(f"  x → {direcao}: assíntota {item['tipo']}: {item['reta']}")
    else:
        print(f"  Nenhuma assíntota horizontal ou oblíqua identificada.")

### Definição de funções e domínio

Defina a função em f no bloco abaixo. Use x como variável independente. Exemplos: x**2 para x², sp.sqrt\(x\) para √x, sp.log\(x\) para ln\(x\) e sp.Piecewise((x**2, x<0), (x, True)) para uma função por partes. Use pi, inf e e para π, infinito e número de Euler, respectivamente.

Uma expressão simplificada pode ocultar restrições do domínio original. Cancelar um fator não torna a função original definida onde o denominador era zero.

In [ ]:
# EXEMPLO DE DEFINIÇÃO
f = (x**2 - 1)/(x - 1)

print(f"Função: {f}")
print(f"Zeros do denominador: {sp.solve(sp.denom(f), x)}")

## 2. Limites, continuidade e assíntotas

### Localize o exercício pelo que ele pede

| O enunciado pede... | Bloco |
|---|---|
| Limite em um ponto, bilateral ou lateral | **2.1** |
| Limite de função definida por partes | **2.2** |
| Limite no infinito de função por partes | **2.3** |
| Continuidade e valor de parâmetro | **2.4** |
| Assíntotas de uma função comum | **2.5** |
| Assíntotas de função por partes | **2.6** |
| Simplificar indeterminação e calcular limite | **2.7** |
| Comparar valores numéricos perto do ponto | **2.8** |

Em cada bloco, altere apenas as linhas sob **ENTRADAS** e execute a célula. As funções auxiliares já estão definidas na configuração.

**Sintaxe:** use ** para potências, inf para +∞ e -inf para −∞. Em sp.Piecewise, cada par tem a forma (expressão, condição); as condições são verificadas na ordem escrita. Use (expressão, True) para o caso restante.

### 2.1 Limite em um ponto

Insira a expressão em f e o ponto de aproximação. Para um limite lateral, defina lado como '-' (esquerda) ou '+' (direita). Deixe lado como None para calcular ambos os limites laterais e o bilateral.

In [ ]:
# ENTRADAS
f = (x**2 - 4)/(x - 2)
ponto = 2
lado = None  # None = bilateral; use '-' ou '+' para limite lateral

# RESULTADOS
if lado in ("-", "+"):
    resultado = sp.limit(f, x, ponto, dir=lado)
    print(f"Limite pela {'esquerda' if lado == '-' else 'direita'}: {resultado}")
else:
    esquerda = sp.limit(f, x, ponto, dir="-")
    direita = sp.limit(f, x, ponto, dir="+")
    print(f"Limite pela esquerda: {esquerda}")
    print(f"Limite pela direita: {direita}")
    print(f"Limite bilateral: {formatar_limite_bilateral(esquerda, direita)}")

### 2.2 Limites laterais de função definida por partes

Defina f usando sp.Piecewise e informe o ponto. O bloco seleciona o ramo usado em cada lado e calcula os limites. O valor da função no ponto é mostrado separadamente, pois não determina os limites laterais.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    (x**2, x <= 2),
    (3*x - 2, x > 2)
)
ponto = 2

# RESULTADOS
esquerda = limite_lateral(f, ponto, "-")
direita = limite_lateral(f, ponto, "+")
print(f"Função: {f}")
print(f"Limite pela esquerda: {esquerda}")
print(f"Limite pela direita: {direita}")
print(f"Limite bilateral: {formatar_limite_bilateral(esquerda, direita)}")
print(f"Valor da função no ponto: {valor_funcao(f, ponto)}")

### 2.3 Limites no infinito de funções por partes

Defina f com sp.Piecewise. O bloco identifica o ramo ativo para valores muito positivos e negativos e calcula os limites em +∞ e −∞.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    ((2*x**2 + 1)/(x**2 + 3), x > 2),
    ((x + 1)/(x - 1), True)
)

# RESULTADOS
for direcao, sinal in ((inf, "+"), (-inf, "-")):
    ramo = ramo_no_infinito(f, sinal)
    print(f"Ramo usado quando x tende a {direcao}: {ramo}")
    print(f"Limite quando x tende a {direcao}: {sp.limit(ramo, x, direcao)}")

### 2.4 Continuidade e parâmetro desconhecido

Para continuidade em x = ponto, os limites laterais e o valor da função nesse ponto devem ser iguais. Insira as expressões válidas à esquerda e à direita e o valor no ponto. Declare parâmetros desconhecidos com sp.symbols.

In [ ]:
# ENTRADAS
k = sp.symbols("k", real=True)
expressao_esquerda = x**2
expressao_direita = 3*x - 2
valor_no_ponto = k
ponto = 2

# RESULTADOS
L_esq = sp.limit(expressao_esquerda, x, ponto, dir="-")
L_dir = sp.limit(expressao_direita, x, ponto, dir="+")
print(f"Limite pela esquerda: {L_esq}")
print(f"Limite pela direita: {L_dir}")
if sp.simplify(L_esq - L_dir) != 0:
    print(f"Continuidade impossível: os limites laterais são diferentes.")
else:
    solucoes = sp.solve(sp.Eq(valor_no_ponto, L_esq), dict=True)
    print(f"Valor exigido para continuidade: {L_esq}")
    print(f"Parâmetro(s) que garantem continuidade: {solucoes if solucoes else "nenhum parâmetro desconhecido foi encontrado"}")

### 2.5 Assíntotas de uma função

Insira a expressão em f. O bloco procura automaticamente candidatos a assíntotas verticais nos zeros reais do denominador e calcula assíntotas horizontais ou oblíquas em cada direção infinita. Descontinuidades removíveis não são classificadas como assíntotas verticais.

In [ ]:
# ENTRADAS
f = (2*x**2 + 1)/(x - 1)

# RESULTADOS
mostrar_assintotas(analisar_assintotas(f))

### 2.6 Assíntotas de funções por partes

Defina f com sp.Piecewise. O bloco detecta pontos de transição nas condições e zeros dos denominadores dos ramos, examina os limites laterais e calcula as assíntotas no infinito. A detecção é voltada a condições usuais com desigualdades e expressões racionais; confira casos com condições simbólicas ou domínios mais complexos.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    ((x**2 + 1)/(x - 2), x > 2),
    ((3*x + 1)/(x + 1), True)
)

# RESULTADOS
mostrar_assintotas(analisar_assintotas(f))

### 2.7 Indeterminações, fatoração e racionalização

Use este bloco quando a substituição direta produzir uma forma como 0/0 ou quando houver radicais a simplificar. Insira f e ponto. Para estudar uma expressão oscilatória, substitua f pela expressão desejada.

In [ ]:
# ENTRADAS
f = (sp.sqrt(x + 1) - 1)/x
ponto = 0

# RESULTADOS
print(f"Expressão original: {f}")
print(f"Forma simplificada: {sp.simplify(f)}")
print(f"Fatoração (quando aplicável): {sp.factor(f)}")
esquerda = sp.limit(f, x, ponto, dir="-")
direita = sp.limit(f, x, ponto, dir="+")
print(f"Limite pela esquerda: {esquerda}")
print(f"Limite pela direita: {direita}")
print(f"Limite bilateral: {formatar_limite_bilateral(esquerda, direita)}")

### 2.8 Conferência numérica perto de um ponto

Informe f, ponto e as distâncias para aproximar x por cada lado. A tabela ajuda a visualizar o comportamento, mas não prova a existência do limite. Distâncias menores aproximam mais o ponto, mas podem aumentar erros numéricos.

In [ ]:
# ENTRADAS
f = (x**2 - 4)/(x - 2)
ponto = 2
distancias = [1e-1, 1e-2, 1e-3, 1e-4]

# RESULTADOS
print(f"Valores de f(x) perto de x = {ponto}:")
print(f"{'Distância':>12} | {'Pela esquerda':>18} | {'Pela direita':>18}")
print(f"{"-" * 55}")
for d in distancias:
    valor_esq = sp.N(f.subs(x, sp.Float(ponto-d)), 10)
    valor_dir = sp.N(f.subs(x, sp.Float(ponto+d)), 10)
    print(f"{d:12.1e} | {str(valor_esq):>18} | {str(valor_dir):>18}")

## 3. Derivadas: cálculo e consulta

Use os blocos conforme o enunciado pedir derivada, derivada de ordem superior, derivada em um ponto, continuidade, derivabilidade, derivação implícita ou função inversa.

| O enunciado pede... | Bloco |
|---|---|
| Primeira, segunda, terceira ou outra ordem de derivada; valor num ponto; simplificação ou fatoração | 3.1 |
| Verificar continuidade, derivadas laterais ou derivabilidade em um ponto | 3.2 |
| Determinar parâmetros de uma função por partes para garantir derivabilidade | 3.2.1 |
| Derivar uma equação com x e y; derivação implícita; derivada da função inversa | 3.3 |

**Sintaxe:** use `x**2` para x², `sp.sin\(x\)` para seno, `sp.cos\(x\)` para cosseno, `sp.exp\(x\)` para exponencial e `sp.log\(x\)` para logaritmo natural. Para frações, use parênteses: `(x**2+1)/(x-1)`.

### 3.1 Derivadas de uma função — todas as ordens em um único bloco

**Use quando o enunciado pedir:** derivada primeira, segunda derivada, terceira derivada, derivada de ordem \(n\), valor da derivada em um ponto, simplificação ou fatoração da derivada.

**ENTRADAS**
- `f`: função que aparece no enunciado.
- `ordem_maxima`: maior ordem solicitada. Ex.: 1 para (f'), 2 para (f''), 3 para (f''').
- `ponto`: valor de x para calcular as derivadas naquele ponto. Deixe `None` se não pedir avaliação pontual.

O bloco calcula as derivadas de todas as ordens até o limite informado. Também apresenta as formas simplificada e fatorada. Se informar um ponto, calcula cada derivada nele. Assim, não é necessário procurar outro bloco para cada ordem.

In [ ]:
# ENTRADAS
f = x**3 - 5*x**2 + 2*x + 7
ordem_maxima = 3
ponto = 2  # Use None se o exercício não pedir derivada em um ponto

# RESULTADOS
if not isinstance(ordem_maxima, int) or ordem_maxima < 1:
    raise ValueError("ordem_maxima deve ser um inteiro maior ou igual a 1")

derivadas = {}
expressao = f

for ordem in range(1, ordem_maxima + 1):
    expressao = sp.diff(expressao, x)
    derivadas[f"ordem_{ordem}"] = {
        "expressao": expressao,
        "simplificada": sp.simplify(expressao),
        "fatorada": sp.factor(expressao),
        "valor_no_ponto": sp.simplify(expressao.subs(x, ponto)) if ponto is not None else "Não solicitado"
    }

print(f"Função original: {f}")
for nome, resultado in derivadas.items():
    print(f"\\nDerivada {nome.split('_')[1]}:")
    print(f"  Expressão:    {resultado["expressao"]}")
    print(f"  Simplificada: {resultado["simplificada"]}")
    print(f"  Fatorada:     {resultado["fatorada"]}")
    if ponto is not None:
        print(f"  Em x = {ponto}: {resultado["valor_no_ponto"]}")

### 3.2 Continuidade e derivabilidade em um ponto

**Use quando o enunciado pedir:** verificar se uma função é contínua ou derivável em determinado ponto; comparar derivadas laterais; identificar ponto anguloso; analisar uma função definida por partes.

**ENTRADAS**
- `g`: função. Para vários trechos, use `sp.Piecewise`.
- `ponto`: ponto analisado.

**Como interpretar:** continuidade exige que os limites laterais coincidam com o valor da função. Derivabilidade exige, além da continuidade, que as derivadas laterais coincidam. Nos extremos do domínio, considere apenas o lado pertencente ao domínio.

In [ ]:
# ENTRADAS
g = sp.Abs(x)
ponto = 0

# RESULTADOS
valor = g.subs(x, ponto)
limite_esquerda = sp.limit(g, x, ponto, dir="-")
limite_direita = sp.limit(g, x, ponto, dir="+")
derivada_esquerda = sp.limit(
    (g - valor) / (x - ponto), x, ponto, dir="-"
)
derivada_direita = sp.limit(
    (g - valor) / (x - ponto), x, ponto, dir="+"
)

continua = (limite_esquerda == valor) and (limite_direita == valor)
derivavel = continua and (derivada_esquerda == derivada_direita)

print(f"Valor da função: {valor}")
print(f"Limite pela esquerda: {limite_esquerda}")
print(f"Limite pela direita: {limite_direita}")
print(f"É contínua no ponto? {continua}")
print(f"Derivada pela esquerda: {derivada_esquerda}")
print(f"Derivada pela direita: {derivada_direita}")
print(f"É derivável no ponto? {derivavel}")

### 3.2.1 Determinar parâmetros para garantir derivabilidade

**Use quando o enunciado fornecer uma função definida por partes com parâmetros desconhecidos** e pedir os valores desses parâmetros para que a função seja derivável no ponto de transição.

**ENTRADAS**
- `g`: função definida com `sp.Piecewise`. Informe os trechos e suas condições.
- `ponto`: ponto em que a expressão da função muda.
- `parametros`: lista das incógnitas que o sistema deve determinar.

**O que o código faz:** identifica os trechos à esquerda e à direita do ponto, monta as condições de continuidade e de igualdade das derivadas laterais e resolve o sistema para os parâmetros informados. Se houver mais de uma solução, todas serão apresentadas. O procedimento considera um ponto de transição interior ao domínio, com trechos diferenciáveis em cada lado.

In [ ]:
# ENTRADAS — função definida por partes
m, p = sp.symbols("m p", real=True)
g = sp.Piecewise(
    (p*x**2 + 2, x < 2),
    (m*x + 1, True)
)

# ENTRADAS — ponto de transição e parâmetros desconhecidos
ponto = 2
parametros = [m, p]  # informe aqui todos os parâmetros que deseja determinar

# RESULTADOS — identificação dos trechos à esquerda e à direita
trecho_esquerdo = ramo_lateral(g, ponto, "-")
trecho_direito = ramo_lateral(g, ponto, "+")

# RESULTADOS — condições de continuidade
valor_no_ponto = g.subs(x, ponto)
limite_esquerda = sp.limit(trecho_esquerdo, x, ponto, dir="-")
limite_direita = sp.limit(trecho_direito, x, ponto, dir="+")

condicoes_continuidade = [
    sp.Eq(limite_esquerda, valor_no_ponto),
    sp.Eq(limite_direita, valor_no_ponto)
]

# RESULTADOS — condição de igualdade das derivadas laterais
derivada_esquerda = sp.limit(sp.diff(trecho_esquerdo, x), x, ponto, dir="-")
derivada_direita = sp.limit(sp.diff(trecho_direito, x), x, ponto, dir="+")

condicao_derivabilidade = sp.Eq(derivada_esquerda, derivada_direita)

# RESULTADOS — resolução do sistema
condicoes = condicoes_continuidade + [condicao_derivabilidade]

# Condições já verdadeiras são removidas; uma condição falsa torna o sistema impossível.
if any(condicao is sp.false for condicao in condicoes):
    solucoes = []
else:
    condicoes_para_resolver = [
        condicao for condicao in condicoes if condicao is not sp.true
    ]
    solucoes = sp.solve(condicoes_para_resolver, parametros, dict=True)

print(f"Trecho à esquerda: {trecho_esquerdo}")
print(f"Trecho à direita: {trecho_direito}")
print(f"\nCONDIÇÕES")
print(f"Continuidade pela esquerda: {condicoes_continuidade[0]}")
print(f"Continuidade pela direita: {condicoes_continuidade[1]}")
print(f"Igualdade das derivadas laterais: {condicao_derivabilidade}")

print(f"\nSOLUÇÕES DOS PARÂMETROS")
if solucoes:
    for indice, solucao in enumerate(solucoes, start=1):
        print(f"Solução {indice}: {solucao}")
else:
    print(f"Nenhuma solução foi encontrada para os parâmetros informados.")


### 3.3 Derivação implícita e relação inversa

**Use quando o enunciado apresentar uma equação que relaciona x e y** e pedir derivadas de y em função de x, derivadas da relação inversa (x em função de y), derivadas de ordem superior ou valores em um ponto.

Informe a relação uma única vez em `F`, colocando todos os termos no lado esquerdo e zero no lado direito. Exemplo: `x**2 + x*y + y**2 = 1` torna-se `F = x**2 + x*y + y**2 - 1`.

O bloco apresenta as expressões simbólicas das derivadas de y em relação a x e da relação inversa, x em relação a y.

**Entradas adicionais (opcionais):**
- `ordem_maxima`: maior ordem da derivada a calcular. Use 1 para a primeira derivada, 2 para a segunda etc.
- `ponto_x`: coordenada x quando o enunciado pedir o valor da derivada em um ponto.
- `ponto_y`: coordenada y do ponto, caso seja fornecida. Se ficar como `None` e `ponto_x` tiver um valor, o bloco tentará determinar y substituindo x na relação original. Se houver mais de uma solução possível, informe também `ponto_y`.

As expressões simbólicas são sempre exibidas primeiro. A avaliação numérica é feita depois, quando `ponto_x` for informado.

In [25]:
# ENTRADAS — relação do enunciado
y = sp.symbols("y", real=True)
F = x**2 + x*y + y**2 - 1

# ENTRADAS — cálculos adicionais (mantenha os valores padrão quando não forem solicitados)
ordem_maxima = 2  # 1 calcula a primeira derivada; 2 calcula até a segunda etc.
ponto_x = None   # coordenada x para avaliação numérica
ponto_y = None   # coordenada y; se None, o bloco tenta obtê-la pela relação F = 0

# RESULTADOS — validação das entradas
if not isinstance(ordem_maxima, int) or isinstance(ordem_maxima, bool) or ordem_maxima < 1:
    raise ValueError("ordem_maxima deve ser um inteiro maior ou igual a 1")

# RESULTADOS — derivadas de y em relação a x
derivadas_diretas = {
    ordem: sp.factor(sp.idiff(F, y, x, n=ordem))
    for ordem in range(1, ordem_maxima + 1)
}

print(f"DERIVADAS DE y EM RELAÇÃO A x")
for ordem, expressao in derivadas_diretas.items():
    simbolo = "dy/dx" if ordem == 1 else f"d^{ordem}y/dx^{ordem}"
    print(f"Ordem {ordem} ({simbolo}): {expressao}")

# RESULTADOS — derivadas da relação inversa, x em função de y
derivadas_inversas = {
    ordem: sp.factor(sp.idiff(F, x, y, n=ordem))
    for ordem in range(1, ordem_maxima + 1)
}

print(f"\nDERIVADAS DE x EM RELAÇÃO A y (RELAÇÃO INVERSA)")
for ordem, expressao in derivadas_inversas.items():
    simbolo = "dx/dy" if ordem == 1 else f"d^{ordem}x/dy^{ordem}"
    print(f"Ordem {ordem} ({simbolo}): {expressao}")

# RESULTADOS — avaliação numérica opcional
if ponto_x is not None:
    if ponto_y is None:
        solucoes_y = sp.solve(F.subs(x, ponto_x), y)
        solucoes_reais = [
            solucao for solucao in solucoes_y
            if solucao.is_real is True
        ]

        if len(solucoes_reais) != 1:
            raise ValueError(
                "Não foi possível determinar um único valor real para y. "
                "Informe também ponto_y."
            )
        ponto_y_calculado = solucoes_reais[0]
    else:
        ponto_y_calculado = ponto_y

    ponto = {x: ponto_x, y: ponto_y_calculado}
    print(f"\nPONTO UTILIZADO: x = {ponto_x}, y = {ponto_y_calculado}")

    print(f"Derivadas de y em relação a x:")
    for ordem, expressao in derivadas_diretas.items():
        print(f"  Ordem {ordem}: {sp.simplify(expressao.subs(ponto))}")

    print(f"Derivadas de x em relação a y:")
    for ordem, expressao in derivadas_inversas.items():
        print(f"  Ordem {ordem}: {sp.simplify(expressao.subs(ponto))}")

DERIVADAS DE y EM RELAÇÃO A x
Ordem 1 (dy/dx): -(2*x + y)/(x + 2*y)
Ordem 2 (d^2y/dx^2): -6*(x**2 + x*y + y**2)/(x + 2*y)**3

DERIVADAS DE x EM RELAÇÃO A y (RELAÇÃO INVERSA)
Ordem 1 (dx/dy): -(x + 2*y)/(2*x + y)
Ordem 2 (d^2x/dy^2): -6*(x**2 + x*y + y**2)/(2*x + y)**3

AVALIAÇÃO NO PONTO (2, 2)
Derivadas de y em relação a x:
  Ordem 1: -1
  Ordem 2: -1/3
Derivadas de x em relação a y:
  Ordem 1: -1
  Ordem 2: -1/3


## 4. Derivadas: aplicações
### Localize o exercício pelo que ele pede

| Tipo de exercício | Bloco |
|---|---|
| Retas tangente e normal | 4.1 |
| Taxas de variação e taxas relacionadas | 4.2 |
| Crescimento, extremos e testes de derivadas | 4.3 |
| Concavidade e inflexão | 4.4 |
| Otimização em intervalo fechado | 4.5 |
| Visualização da análise de uma função | 4.6 |

Altere as expressões e os valores indicados em cada célula de código e execute para obter os resultados.


**Como identificar o procedimento:** inclinação ou tangente → `f'\(a\)`; normal → inclinação `-1/f'\(a\)` quando `f'\(a\)≠0`; velocidade → derivada da posição; aceleração → segunda derivada; extremos → pontos críticos e comparação; concavidade → sinal de `f''`; otimização → modelar objetivo, domínio e restrições.

A derivada fornece candidatos, **não** garante extremos. Pontos críticos interiores ocorrem onde `f'=0` ou onde `f'` não existe (com `f` definida).

### 4.1 Retas tangente e normal

**Use quando o exercício pedir a equação da reta tangente, da reta normal, a inclinação ou o coeficiente angular de uma curva em um ponto.** Informe a função e a coordenada x do ponto.

A inclinação da tangente é \(f'\(a\)\), e a reta tangente passa por \((a,f\(a\))\). Quando \(f'\(a\)\ne 0\), a inclinação da normal é \(-1/f'\(a\)\). Se a tangente for horizontal, a normal é vertical, dada por \(x=a\). Se a derivada não existir, analise a geometria local antes de aplicar as fórmulas.

In [ ]:
# ENTRADAS — função e ponto de tangência
f = x**2 + 1
a = sp.Integer(2)

# RESULTADOS
inclinacao = sp.diff(f, x).subs(x, a)
ponto_y = f.subs(x, a)
tangente = sp.expand(ponto_y + inclinacao*(x-a))
print(f"Ponto de tangência: ({a}, {ponto_y})")
print(f"Inclinação da tangente: {inclinacao}")
print(f"Equação da reta tangente: y = {tangente}")
if inclinacao == 0:
    print(f"Equação da reta normal: x = {a} {"(normal vertical)"}")
else:
    normal = sp.expand(ponto_y - (x-a)/inclinacao)
    print(f"Equação da reta normal: y = {normal}")

### 4.2 Taxas de variação, movimento e taxas relacionadas

Esta seção contempla exercícios de velocidade, aceleração, posição, taxa instantânea de variação e taxas relacionadas entre grandezas. Use o bloco 4.2.1 para movimento descrito por uma função posição-tempo e o bloco 4.2.2 para relações entre grandezas, como raio e área de um círculo.

#### 4.2.1 Movimento: posição, velocidade e aceleração

**Use quando o enunciado fornecer a posição em função do tempo e pedir velocidade instantânea, aceleração ou taxa de variação da posição.** Informe \(s(t)\); o bloco calcula \(v(t)=s'(t)\) e \(a(t)=s''(t)\).

In [ ]:
# ENTRADAS — função posição em relação ao tempo
t = sp.symbols("t",real=True)
posicao = 2*t**3-9*t**2+12*t

# RESULTADOS — velocidade e aceleração
velocidade = sp.diff(posicao,t)
aceleracao = sp.diff(posicao,t,2)
print(f"Posição s(t): {posicao}")
print(f"Velocidade v(t) = s'(t): {velocidade}")
print(f"Aceleração a(t) = v'(t): {aceleracao}")

#### 4.2.2 Taxas relacionadas entre grandezas

**Use quando o enunciado relacionar grandezas que variam com o tempo**, como lado e área de um quadrado, raio e área de um círculo, raio e volume de uma esfera ou distância entre objetos.

Escolha uma expressão do catálogo expressoes ou escreva uma expressão manualmente. A variável independente é a grandeza que aparece na expressão. Depois, informe o valor dessa grandeza e sua taxa de variação no instante analisado.

Há duas formas de fornecer os dados:
- **Valores diretos:** preencha valor e taxa. Deixe funcao_tempo e instante como None.
- **Função temporal:** preencha funcao_tempo e instante. O bloco calcula automaticamente o valor e a taxa naquele instante; nesse modo, valor e taxa são ignorados.

A função calcular_taxa_relacionada(), definida na configuração inicial, aplica a regra da cadeia. Altere apenas as entradas do exercício.

In [ ]:
# EXPRESSÃO MATEMÁTICA — escolha do catálogo ou definição manual
expressao = expressoes.quadrado.area
variavel = x

# DADOS DO EXERCÍCIO — use valor e taxa quando forem fornecidos diretamente
valor = 10
taxa = 3

# FUNÇÃO DO TEMPO — opcional; quando preenchida, substitui valor e taxa
funcao_tempo = None  # Ex.: 2*t + 1
instante = None      # Ex.: 5

# CÁLCULO
resultado_taxa = calcular_taxa_relacionada(expressao, variavel, valor, taxa, funcao_tempo, instante)

# RESULTADOS
variavel_tempo = sp.Function(str(variavel))(t)
if funcao_tempo is None:
    derivada_tempo = sp.diff(expressao.subs(variavel, variavel_tempo), t)
else:
    derivada_tempo = sp.diff(expressao.subs(variavel, funcao_tempo), t)
print(f"Expressão original: {sp.Eq(sp.Symbol('y'), expressao)}")
print(f"Derivada em relação ao tempo: {derivada_tempo}")
if funcao_tempo is None:
    print(f"Valor utilizado para {variavel}: {resultado_taxa['valor']}")
    print(f"Taxa de variação de {variavel}: {resultado_taxa['taxa']}")
    if instante is not None:
        print(f"Instante analisado: t = {instante}")
else:
    print(f"Função temporal utilizada: {variavel}(t) = {funcao_tempo}")
    print(f"Instante analisado: t = {instante}")
    print(f"Valor calculado de {variavel}: {resultado_taxa['valor']}")
    print(f"Taxa calculada de {variavel}: {resultado_taxa['taxa']}")
print(f"Taxa de variação da expressão em relação ao tempo: {resultado_taxa['resultado']}")

### 4.3 Crescimento, decrescimento, pontos críticos e extremos

**Use quando o enunciado pedir intervalos de crescimento ou decrescimento, pontos críticos, máximos e mínimos locais, ou classificação de extremos pelo teste da primeira ou da segunda derivada.**

O sinal de \(f'(x)\) determina crescimento e decrescimento. Nos pontos críticos com \(f'(c)=0\), o teste da segunda derivada classifica mínimo local se \(f''(c)>0\), máximo local se \(f''(c)<0\), e é inconclusivo se \(f''(c)=0\). O bloco também resolve as desigualdades \(f'(x)>0\) e \(f'(x)<0\). Informe separadamente pontos onde a função é definida, mas a derivada não existe, pois eles também podem ser críticos.

In [ ]:
# ENTRADAS — função
f = x**3 - 6*x**2 + 9*x + 1

# ENTRADAS — pontos críticos onde a derivada não existe (se houver)
pontos_nao_diferenciaveis = []

# RESULTADOS
primeira = sp.diff(f,x)
segunda = sp.diff(f,x,2)
criticos_estacionarios = sp.solve(sp.Eq(primeira,0),x)
criticos = list(criticos_estacionarios)+list(pontos_nao_diferenciaveis)
print(f"Primeira derivada: {primeira}")
print(f"Segunda derivada: {segunda}")
print(f"\nPONTOS CRÍTICOS")
if not criticos:
    print(f"Nenhum ponto crítico informado ou encontrado.")
for c in criticos:
    if c in criticos_estacionarios:
        teste = segunda.subs(x,c)
        tipo = ("mínimo local" if teste.is_positive else
                "máximo local" if teste.is_negative else "teste da segunda derivada inconclusivo")
        print(f"x = {c}: f(x) = {f.subs(x,c)}; f''(x) = {teste}; {tipo}.")
    else:
        print(f"x = {c}: f(x) = {f.subs(x,c)}; derivada não existe — classifique pelo comportamento de f' nos lados.")
print(f"\nINTERVALOS")
print(f"Crescimento: {sp.solve_univariate_inequality(primeira>0,x,relational=False)}")
print(f"Decrescimento: {sp.solve_univariate_inequality(primeira<0,x,relational=False)}")

### 4.4 Concavidade e pontos de inflexão

**Use quando o exercício pedir concavidade para cima ou para baixo, intervalos de concavidade, candidatos a ponto de inflexão ou pontos de inflexão.**

O sinal da segunda derivada \(f''\(x\)\) determina a concavidade: positiva indica concavidade para cima e negativa indica concavidade para baixo. Resolver \(f''\(x\)=0\) fornece apenas candidatos. Para confirmar um ponto de inflexão, verifique se a concavidade realmente muda ao atravessar o ponto.

In [ ]:
# ENTRADAS — função
f = x**3 - 6*x**2 + 9*x + 1

# RESULTADOS
segunda = sp.diff(f,x,2)
candidatos = sp.solve(sp.Eq(segunda,0),x)
print(f"Segunda derivada: {segunda}")
print(f"Candidatos a pontos de inflexão: {candidatos}")
print(f"Concavidade para cima (f'' > 0): {sp.solve_univariate_inequality(segunda>0,x,relational=False)}")
print(f"Concavidade para baixo (f'' < 0): {sp.solve_univariate_inequality(segunda<0,x,relational=False)}")
print(f"Confirme a mudança de sinal de f'' em cada candidato antes de classificá-lo como ponto de inflexão.")

### 4.5 Problemas de otimização e extremos absolutos em intervalo fechado

**Use para maximizar ou minimizar áreas, volumes, custos, receitas, distâncias ou outras funções objetivo sujeitas a restrições e intervalos.**

Informe a função objetivo e os limites do intervalo admissível. O bloco calcula os pontos estacionários interiores, acrescenta os extremos do intervalo e compara os valores da função. Se houver pontos não diferenciáveis no interior, inclua-os na lista de candidatos. A modelagem da situação — domínio, restrições e unidades — deve ser feita antes do cálculo.

In [ ]:
# ENTRADAS — função objetivo e intervalo admissível
funcao_objetivo = x*(10-x)
limite_inferior = 0
limite_superior = 10

# ENTRADAS — pontos não diferenciáveis no interior (se houver)
pontos_nao_diferenciaveis = []

# RESULTADOS — pontos estacionários, pontos não diferenciáveis e extremos
derivada = sp.diff(funcao_objetivo,x)
criticos = sp.solve(sp.Eq(derivada,0),x)
candidatos = [sp.sympify(limite_inferior),sp.sympify(limite_superior)]
for c in criticos + pontos_nao_diferenciaveis:
    if c.is_real is True and bool(c>=limite_inferior) and bool(c<=limite_superior):
        candidatos.append(c)
candidatos = list(dict.fromkeys(candidatos))
valores = [(c,sp.simplify(funcao_objetivo.subs(x,c))) for c in candidatos]
maximo = max(valores,key=lambda item:float(sp.N(item[1])))
minimo = min(valores,key=lambda item:float(sp.N(item[1])))
print(f"Função objetivo: {funcao_objetivo}")
print(f"Derivada: {derivada}")
print(f"Candidatos (x, f(x)): {valores}")
print(f"Máximo global no intervalo: {maximo}")
print(f"Mínimo global no intervalo: {minimo}")

### 4.6 Visualização da análise de uma função

Altere `f` e o intervalo de visualização para estudar outros exemplos. O gráfico ajuda a interpretar, mas não substitui a verificação algébrica de domínio, sinais e extremos.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x**3-6*x**2+9*x+1
xs = np.linspace(-0.5,4.5,500)
fn = sp.lambdify(x,f,'numpy')
criticos = [float(c) for c in sp.solve(sp.Eq(sp.diff(f,x),0),x) if c.is_real]
inflexoes = [float(c) for c in sp.solve(sp.Eq(sp.diff(f,x,2),0),x) if c.is_real]
plt.figure(figsize=(9,5))
plt.plot(xs,fn(xs),label='f(x)')
for c in criticos:
    plt.scatter([c],[float(f.subs(x,c))],marker='o',s=65,label=f'crítico x={c:g}')
for c in inflexoes:
    plt.scatter([c],[float(f.subs(x,c))],marker='x',s=90,label=f'inflexão x={c:g}')
plt.axhline(0,color='gray',linewidth=.7)
plt.grid(True);plt.legend();plt.xlabel('x');plt.ylabel('f(x)');plt.show()

## 5. Integrais: conceitos, propriedades e técnicas
### Localize o exercício pelo que ele pede

| Tipo de exercício | Bloco |
|---|---|
| Integrais imediatas e propriedades | 5.1 |
| Soma de Riemann e integral definida | 5.2 |
| Teorema Fundamental do Cálculo (TFC) | 5.3 |
| Substituição de variável | 5.4 |
| Integração por partes | 5.5 |
| Frações parciais | 5.6 |
| Verificação e limitações | 5.7 |

Altere as expressões e os valores indicados em cada célula de código e execute para obter os resultados.


A **integral indefinida** fornece uma família de primitivas: se `F'=f`, então `∫f\(x\)dx=F\(x\)+C`. Uma **condição inicial** determina `C`. A **integral definida** produz um valor numérico (com sinal), interpretado como limite de somas de Riemann quando a função é integrável.

**Escolha do procedimento:** integral imediata → `sp.integrate`; função composta com derivada interna → substituição; produto de funções → partes; função racional → divisão polinomial e frações parciais. Verifique a derivada da primitiva e respeite o domínio.

### Primitiva com condição inicial

**Use quando o exercício pedir determinar a constante de integração a partir de uma condição inicial**, como \(F\(a\)=b\), ou encontrar a primitiva particular que passa por um ponto. Informe o integrando e o valor da função no ponto conhecido.

In [ ]:
# ENTRADAS — integrando e condição inicial F(ponto_condicao) = valor_condicao
integrando = 3*x**2 + 2*x
ponto_condicao = 1
valor_condicao = 5

# RESULTADOS
primitiva_geral = sp.integrate(integrando,x)
C = sp.symbols("C")
solucao_C = sp.solve(sp.Eq((primitiva_geral+C).subs(x,ponto_condicao),valor_condicao),C)
print(f"Integrando: {integrando}")
print(f"Família de primitivas: F(x) = {primitiva_geral} {"+ C"}")
if solucao_C:
    print(f"Constante determinada pela condição inicial: {solucao_C[0]}")
    print(f"Primitiva particular: F(x) = {primitiva_geral+solucao_C[0]}")
else:
    print(f"Não foi possível determinar a constante com a condição informada.")

### 5.1 Integrais imediatas, primitivas e propriedades de linearidade

**Use quando o enunciado pedir uma integral indefinida, antiderivada, primitiva ou integração direta de polinômios, potências, exponenciais, logaritmos ou funções trigonométricas.**

Informe um ou vários integrandos. O bloco calcula uma primitiva para cada expressão e verifica o resultado derivando-a. Em integrais indefinidas, acrescente a constante de integração (+C). A linearidade permite separar somas e retirar constantes multiplicativas.

In [ ]:
# ENTRADAS — informe um ou vários integrandos
integrandos = [x**4,1/x,sp.exp(x),sp.sin(x),sp.cos(x),2*x+sp.cos(x)]

# RESULTADOS
for integrando in integrandos:
    primitiva = sp.integrate(integrando,x)
    verificacao = sp.simplify(sp.diff(primitiva,x)-integrando)
    print(f"\nIntegrando: {integrando}")
    print(f"Primitiva: {primitiva} {"+ C"}")
    print(f"Verificação (derivada da primitiva menos integrando): {verificacao}")

### 5.2 Soma de Riemann, aproximação de área e integral definida

**Use quando o exercício pedir soma de Riemann à esquerda ou à direita, partição do intervalo, largura \(\Delta x\), limite da soma ou interpretação da integral definida.**

Informe a função, os extremos \(a\) e \(b\), e o número de subintervalos simbólico \(n\). O bloco monta a soma à direita, calcula seu limite quando \(n\to\infty\) e compara com a integral definida. A integral representa área com sinal; regiões abaixo do eixo contribuem negativamente.

In [ ]:
# ENTRADAS — função e intervalo da soma de Riemann
f = x**2
a,b = sp.Integer(0),sp.Integer(2)
n,i = sp.symbols("n i",integer=True,positive=True)

# RESULTADOS — soma à direita, limite e integral definida
dx = (b-a)/n
soma = sp.summation(f.subs(x,a+i*dx)*dx,(i,1,n))
limite_soma = sp.limit(soma,n,sp.oo)
integral_exata = sp.integrate(f,(x,a,b))
print(f"Função: {f}")
print(f"Intervalo: [{a}, {b}]")
print(f"Soma de Riemann à direita: {sp.factor(soma)}")
print(f"Limite da soma quando n tende ao infinito: {limite_soma}")
print(f"Integral definida: {integral_exata}")

### 5.3 Teorema Fundamental do Cálculo (TFC), função acumulada e limites variáveis

**Use quando o enunciado pedir derivar uma integral com limite variável, calcular uma função acumulada, avaliar uma integral definida por uma primitiva ou aplicar o Teorema Fundamental do Cálculo.**

Informe o integrando e os limites. O bloco calcula a função acumulada, sua derivada, a integral definida e a derivada de uma integral cujo limite superior depende de \(x\). Para limites variáveis diferentes de \(x\), a regra da cadeia deve ser considerada.

In [ ]:
# ENTRADAS — integrando, limites e limite superior variável
t = sp.symbols("t",real=True)
integrando = t**2+sp.sin(t)
limite_inferior = 0
limite_superior = 2
limite_variavel = x**2

# RESULTADOS — Teorema Fundamental do Cálculo
funcao_acumulada = sp.integrate(integrando,(t,limite_inferior,x))
integral_definida = sp.integrate(integrando,(t,limite_inferior,limite_superior))
derivada_acumulada = sp.diff(funcao_acumulada,x)
derivada_limite_variavel = sp.diff(sp.integrate(integrando,(t,limite_inferior,limite_variavel)),x)
print(f"Função acumulada G(x): {funcao_acumulada}")
print(f"Derivada G'(x): {sp.simplify(derivada_acumulada)}")
print(f"Integral definida de {limite_inferior} a {limite_superior}: {integral_definida}")
print(f"Derivada da integral com limite superior {limite_variavel}: {sp.simplify(derivada_limite_variavel)}")

### 5.4 Integração por substituição, mudança de variável e troca de limites

**Use quando o integrando contiver uma função composta e a derivada da expressão interna também aparecer**, como em integrais do tipo \(f(g\(x\))g'\(x\)\).

Informe a substituição \(u=g\(x\)\) e o integrando já escrito em função de \(u\). O bloco calcula \(du/dx\), reconstrói o integrando em \(x\), encontra a primitiva e transforma os limites de integração para a variável \(u\). Confira se o fator \(g'\(x\)\) está presente no integrando original.

In [ ]:
# ENTRADAS — substituição u = expressão_u e integrando escrito em u
u = sp.symbols("u",real=True)
expressao_u = x**2
integrando_em_u = sp.cos(u)

# ENTRADAS — limites da integral definida em x
a,b = 0,sp.sqrt(sp.pi)

# RESULTADOS — mudança de variável e integração
du_dx = sp.diff(expressao_u,x)
integrando_em_x = integrando_em_u.subs(u,expressao_u)*du_dx
primitiva_u = sp.integrate(integrando_em_u,u)
primitiva_x = primitiva_u.subs(u,expressao_u)
limite_u_a = expressao_u.subs(x,a)
limite_u_b = expressao_u.subs(x,b)
integral_em_u = sp.integrate(integrando_em_u,(u,limite_u_a,limite_u_b))
print(f"Substituição: {u} {"="} {expressao_u}")
print(f"Derivada da substituição: {du_dx}")
print(f"Integrando correspondente em x: {integrando_em_x}")
print(f"Primitiva em u: {primitiva_u} {"+ C"}")
print(f"Primitiva em x: {primitiva_x} {"+ C"}")
print(f"Verificação pela derivada: {sp.simplify(sp.diff(primitiva_x,x)-integrando_em_x)}")
print(f"Limites transformados: u({a}) = {limite_u_a}; u({b}) = {limite_u_b}")
print(f"Integral definida após a substituição: {integral_em_u}")

### 5.5 Integração por partes, produto de funções e escolha de u e dv

**Use quando o exercício apresentar um produto de funções e pedir integração por partes**, especialmente produtos de polinômios com exponenciais, logaritmos ou funções trigonométricas.

Informe \(u\) e \(dv\), de modo que o integrando seja \(u\,dv\). O código calcula \(du\), a primitiva \(v\) e aplica \(\int u\,dv=uv-\int v\,du\). Escolha \(u\) para que sua derivada simplifique a integral restante.

In [ ]:
# ENTRADAS — escolha u e dv para o integrando u·dv
u = x
dv = sp.exp(x)

# RESULTADOS — integração por partes
du = sp.diff(u,x)
v = sp.integrate(dv,x)
integrando = u*dv
primitiva = sp.simplify(u*v-sp.integrate(v*du,x))
print(f"Integrando: {integrando}")
print(f"u: {u}")
print(f"dv: {dv} {"→ v ="} {v}")
print(f"du: {du} {"dx"}")
print(f"Primitiva: {primitiva} {"+ C"}")
print(f"Verificação: {sp.simplify(sp.diff(primitiva,x)-integrando)}")

### 5.6 Frações parciais, decomposição de funções racionais e integração

**Use quando o exercício pedir decompor ou integrar uma função racional \(P\(x\)/Q\(x\)\) por frações parciais.**

Informe o quociente de polinômios. O bloco calcula a decomposição com `sp.apart`, integra a expressão e verifica a primitiva. Se o grau do numerador for maior ou igual ao do denominador, faça primeiro a divisão polinomial; o exemplo também mostra como obter quociente e resto.

In [ ]:
# ENTRADAS — função racional P(x)/Q(x)
racional = (2*x+3)/(x**2+3*x+2)

# RESULTADOS — decomposição em frações parciais e integração
decomposicao = sp.apart(racional,x)
primitiva = sp.integrate(decomposicao,x)
numerador_improprio = x**3+1
denominador_improprio = x**2-1
quociente,resto = sp.div(numerador_improprio,denominador_improprio,x)
print(f"Função racional: {racional}")
print(f"Decomposição em frações parciais: {decomposicao}")
print(f"Primitiva: {primitiva} {"+ C"}")
print(f"Verificação pela derivada: {sp.simplify(sp.diff(primitiva,x)-racional)}")
print(f"\nPara uma fração imprópria:")
print(f"Quociente da divisão polinomial: {quociente}")
print(f"Resto da divisão polinomial: {resto}")

### 5.7 Verificação de primitivas, integrais não elementares e aproximação numérica

**Use quando o SymPy não encontrar uma primitiva elementar, quando a integral definida exigir aproximação numérica ou quando for necessário conferir o resultado.**

Informe o integrando e os limites. O bloco apresenta a primitiva, verifica sua derivada e calcula a integral definida exata quando possível, além de uma aproximação numérica. Uma expressão integral não avaliada não significa necessariamente erro: pode não existir uma forma fechada elementar.

In [ ]:
# ENTRADAS — integrando e intervalo para avaliação numérica
integrando = x**4-3*x+sp.cos(x)
limite_inferior,limite_superior = 0,1

# RESULTADOS — primitiva, integral definida e aproximação
primitiva = sp.integrate(integrando,x)
integral_exata = sp.integrate(integrando,(x,limite_inferior,limite_superior))
integral_aproximada = sp.N(integral_exata,12)
print(f"Primitiva: {primitiva} {"+ C"}")
print(f"Verificação da primitiva: {sp.simplify(sp.diff(primitiva,x)-integrando)}")
print(f"Integral definida de {limite_inferior} a {limite_superior}: {integral_exata}")
print(f"Valor numérico aproximado: {integral_aproximada}")
print(f"Uma integral não avaliada pode indicar que o SymPy não encontrou uma forma fechada elementar.")

## 6. Integrais: aplicações
### Localize o exercício pelo que ele pede

| Tipo de exercício | Bloco |
|---|---|
| Área líquida, área geométrica e área entre curvas | 6.1 |
| Comprimento de arco | 6.2 |
| Área de superfície de revolução | 6.3 |
| Volumes por seções, discos e arruelas | 6.4 |
| Cascas cilíndricas e rotação em torno de outro eixo | 6.5 |
| Aproximação numérica e checagens | 6.6 |

Altere as expressões e os valores indicados em cada célula de código e execute para obter os resultados.


**Mapa rápido:** área com sinal → `∫f`; área geométrica → `∫|f|`; área entre curvas → `∫|f-g|`; comprimento de arco → `∫sqrt(1+(f')²)`; área de superfície de revolução → `2π∫(raio)ds`; volume por seções → `∫A\(x\)dx`; sólidos de revolução → discos, arruelas ou cascas cilíndricas.

Antes de integrar, **desenhe/identifique a região, o intervalo, o eixo de rotação e as unidades**. Divida intervalos quando curvas se cruzam ou mudam de sinal.

### 6.1 Área líquida, área geométrica, área entre curvas e região limitada

**Use quando o enunciado pedir área sob uma curva, área entre o gráfico e o eixo x, área entre duas curvas, região limitada ou área total.**

A integral definida de \(f\) fornece área com sinal. Para área geométrica entre o gráfico e o eixo, integre \(|f\(x\)|\). Para área entre curvas, integre \(|f\(x\)-g\(x\)|\). Informe os intervalos corretos; quando as curvas se cruzarem, o valor absoluto considera a troca de posição.

In [ ]:
# ENTRADAS — área entre gráfico e eixo x
f = x-1
a,b = 0,3

# ENTRADAS — área entre duas curvas no intervalo
curva_1 = x
curva_2 = x**2
c,d = 0,1

# RESULTADOS
area_liquida = sp.integrate(f,(x,a,b))
area_geometrica = sp.integrate(sp.Abs(f),(x,a,b))
area_entre_curvas = sp.integrate(sp.Abs(curva_1-curva_2),(x,c,d))
print(f"Integral com sinal no intervalo: {area_liquida}")
print(f"Área geométrica entre f(x) e o eixo x: {area_geometrica}")
print(f"Interseções entre as curvas: {sp.solve(sp.Eq(curva_1,curva_2),x)}")
print(f"Área geométrica entre as curvas no intervalo: {area_entre_curvas}")
print(f"O valor absoluto considera a troca de posição das curvas ou a mudança de sinal dentro do intervalo.")

### 6.2 Comprimento de arco, comprimento de curva e curvas paramétricas

**Use quando o exercício pedir comprimento de uma curva, comprimento de arco ou comprimento de uma trajetória parametrizada.**

Para \(y=f\(x\)\), o elemento de arco é \(ds=\sqrt{1+[f'\(x\)]^2}\,dx\). Para uma curva paramétrica \((x(t),y(t))\), use \(ds=\sqrt{[x'(t)]^2+[y'(t)]^2}\,dt\). Informe a função ou as equações paramétricas e os limites correspondentes.

In [ ]:
# ENTRADAS — curva y=f(x) e intervalo
f = x**sp.Rational(3,2)
a,b = 0,4

# ENTRADAS — curva paramétrica x(t), y(t) e intervalo de t
t = sp.symbols("t",real=True)
x_t,y_t = t,t**2
inicio_t,fim_t = 0,1

# RESULTADOS — comprimento de arco
elemento_arco = sp.sqrt(1+sp.diff(f,x)**2)
comprimento = sp.integrate(elemento_arco,(x,a,b))
elemento_parametrico = sp.sqrt(sp.diff(x_t,t)**2+sp.diff(y_t,t)**2)
comprimento_parametrico = sp.integrate(elemento_parametrico,(t,inicio_t,fim_t))
print(f"Elemento de arco ds/dx: {elemento_arco}")
print(f"Comprimento de y=f(x) em [{a}, {b}]: {sp.simplify(comprimento)}")
print(f"Elemento de arco na parametrização: {elemento_parametrico}")
print(f"Comprimento da curva paramétrica em [{inicio_t}, {fim_t}]: {comprimento_parametrico}")

### 6.3 Área de superfície de revolução em torno dos eixos x ou y

**Use quando o enunciado pedir área da superfície gerada pela rotação de uma curva em torno do eixo x ou do eixo y.**

Informe a curva \(y=f\(x\)\), o intervalo e o eixo de rotação. A fórmula é \(S=2\pi\int (raio)\,ds\): ao redor do eixo x, o raio é a distância \(|f\(x\)|\); ao redor do eixo y, é \(|x|\). O resultado é a área lateral da superfície, sem incluir tampas.

In [ ]:
# ENTRADAS — curva y=f(x), intervalo e eixo de rotação
f = x
a,b = 0,2
eixo = "x"  # "x" para o eixo x; "y" para o eixo y

# RESULTADOS — área lateral da superfície de revolução
ds = sp.sqrt(1+sp.diff(f,x)**2)
if eixo == "x":
    integrando_superficie = sp.Abs(f)*ds
elif eixo == "y":
    integrando_superficie = sp.Abs(x)*ds
else:
    raise ValueError("eixo deve ser 'x' ou 'y'.")
superficie = sp.simplify(2*sp.pi*sp.integrate(integrando_superficie,(x,a,b)))
print(f"Curva: {f}")
print(f"Área lateral ao girar em torno do eixo {eixo}: {superficie}")
print(f"O resultado não inclui tampas ou outras superfícies planas do sólido.")

### 6.4 Volume de sólidos por seções transversais, discos e arruelas

**Use quando o exercício pedir volume de sólido, método dos discos, método das arruelas ou volume por seções transversais.**

Para seções perpendiculares ao eixo de integração, informe a área \(A\(x\)\) e calcule \(V=\int A\(x\)\,dx\). Em sólidos de revolução, discos usam \(\pi R\(x\)^2\), enquanto arruelas usam \(\pi[R\(x\)^2-r\(x\)^2]\). Os raios são distâncias ao eixo: devem ser não negativos e o raio externo deve ser maior ou igual ao interno.

In [ ]:
# ENTRADAS — método dos discos: raio externo
raio_disco = x
a_disco,b_disco = 0,2

# ENTRADAS — método das arruelas: raios externo e interno
raio_externo,raio_interno = 2,x
a_arruela,b_arruela = 0,2

# ENTRADAS — método das seções: área de cada seção transversal
area_secao = x**2
a_secao,b_secao = 0,2

# RESULTADOS
volume_discos = sp.pi*sp.integrate(raio_disco**2,(x,a_disco,b_disco))
volume_arruelas = sp.pi*sp.integrate(raio_externo**2-raio_interno**2,(x,a_arruela,b_arruela))
volume_secoes = sp.integrate(area_secao,(x,a_secao,b_secao))
print(f"Volume pelo método dos discos: {volume_discos}")
print(f"Volume pelo método das arruelas: {volume_arruelas}")
print(f"Volume por seções transversais: {volume_secoes}")
print(f"Confira se os raios são distâncias não negativas e se o raio externo é maior ou igual ao interno.")

### 6.5 Método das cascas cilíndricas e rotação em torno de eixo vertical

**Use quando o exercício pedir volume de sólido de revolução pelo método das cascas cilíndricas**, especialmente ao girar uma região em torno do eixo y ou de uma reta vertical \(x=c\).

Cada casca tem raio igual à distância ao eixo de rotação e altura igual à diferença entre as curvas que limitam a região. O volume é \(V=2\pi\int (raio)(altura)\,dx\). Escolha o intervalo de modo que a região não seja contada mais de uma vez.

In [ ]:
# ENTRADAS — região sob y=f(x), intervalo e eixo vertical x=c
f = x
a,b = 0,2
c = 0

# RESULTADOS — método das cascas cilíndricas
raio_casca = sp.Abs(x-c)
altura_casca = f
volume_cascas = 2*sp.pi*sp.integrate(raio_casca*altura_casca,(x,a,b))
print(f"Raio das cascas: {raio_casca}")
print(f"Altura das cascas: {altura_casca}")
print(f"Volume por cascas ao redor de x = {c}: {sp.simplify(volume_cascas)}")
print(f"Defina a região e o intervalo para evitar contar a mesma parte do sólido mais de uma vez.")

### 6.6 Integração numérica, regra dos trapézios e erro de aproximação

**Use quando a integral definida não puder ser calculada facilmente de forma simbólica, quando o enunciado pedir aproximação numérica ou quando for necessário estimar o erro.**

Informe a função, o intervalo e o número de subintervalos. O bloco compara o valor exato obtido pelo SymPy com a aproximação pela regra dos trapézios e calcula o erro absoluto. A precisão depende da suavidade da função e da quantidade de subintervalos.

In [ ]:
# ENTRADAS — função, intervalo e número de subintervalos
f = x**2
a,b = 0,2
numero_subintervalos = 1000

# RESULTADOS
def regra_trapezios(funcao_numerica,inicio,fim,n=1000):
    pontos = np.linspace(inicio,fim,n+1)
    valores = np.asarray(funcao_numerica(pontos),dtype=float)
    return np.trapezoid(valores,pontos) if hasattr(np,"trapezoid") else np.trapz(valores,pontos)

funcao_numerica = sp.lambdify(x,f,"numpy")
valor_exato = sp.integrate(f,(x,a,b))
valor_aproximado = regra_trapezios(funcao_numerica,a,b,numero_subintervalos)
print(f"Integral definida exata: {valor_exato}")
print(f"Aproximação pela regra dos trapézios: {valor_aproximado}")
print(f"Erro absoluto: {abs(valor_aproximado-float(valor_exato))}")

## 7. Escolha rápida da ferramenta

| Objetivo | Código típico |
|---|---|
| Limite simbólico | `sp.limit(expr, x, a)` |
| Limite lateral | `sp.limit(expr, x, a, dir='+' ou '-')` |
| Limite no infinito | `sp.limit(expr, x, sp.oo)` ou `-sp.oo` |
| Simplificar/fatorar | `sp.simplify(expr)`, `sp.factor(expr)` |
| Derivar | `sp.diff(expr, x)` |
| Derivada de ordem 2 | `sp.diff(expr, x, 2)` |
| Integrar sem limites | `sp.integrate(expr, x)` |
| Integral definida | `sp.integrate(expr, (x, a, b))` |
| Avaliar expressão | `expr.subs(x, valor)` |
| Gerar função numérica | `sp.lambdify(x, expr, 'numpy')` |

## Erros frequentes
- Tratar `0/0` como se fosse o valor do limite.
- Supor que o limite existe sem comparar os laterais.
- Confundir `f\(a\)` com `lim f\(x\)`.
- Concluir que toda descontinuidade gera assíntota vertical.
- Esquecer o domínio (por exemplo, `ln\(x\)` exige `x>0` nos reais).
- Em otimização, considerar apenas `f'=0` e ignorar extremos do intervalo ou pontos não diferenciáveis.
- Interpretar integral definida como área geométrica sem considerar o sinal.
- Esquecer `+C` em integrais indefinidas.

## Exercícios rápidos para testar o fluxo

Execute as células e confira se os resultados fazem sentido. Tente prever o resultado antes de executar.

1. Calcule `lim (x**2 - 4)/(x - 2)` quando `x→2`.
2. Verifique se `f\(x\)=(x**2-1)/(x-1)` tem limite em `x=1` e se está definida nesse ponto.
3. Encontre a derivada de `x**2*sin\(x\)`.
4. Calcule `∫_0^3 2x dx`.
5. Calcule a área entre `y=x` e `y=x**2` em `[0,1]`.

In [ ]:
# EXERCÍCIO 1 — limite de (x² - 4)/(x - 2) quando x tende a 2
limite_1 = sp.limit((x**2-4)/(x-2),x,2)
print(f"Exercício 1 — limite: {limite_1}")

# EXERCÍCIO 2 — limite e valor da função em x = 1
funcao_2 = (x**2-1)/(x-1)
limite_2 = sp.limit(funcao_2,x,1)
valor_2 = funcao_2.subs(x,1)
print(f"\nExercício 2 — limite quando x tende a 1: {limite_2}")
valor_2_formatado = "não definido" if valor_2.has(sp.nan, sp.zoo) else valor_2
print(f"Valor da função em x = 1: {valor_2_formatado}")

### Exercícios rápidos: derivadas e integrais

O bloco abaixo resolve os exercícios 3 a 5 da lista anterior: derivada de uma função trigonométrica, integral definida e área entre curvas. Altere as expressões e os limites diretamente nas entradas correspondentes.

In [ ]:
# EXERCÍCIO 3 — derivada de x²·sen(x)
derivada_3 = sp.diff(x**2*sp.sin(x),x)
print(f"Exercício 3 — derivada: {derivada_3}")

# EXERCÍCIO 4 — integral definida de 2x entre 0 e 3
integral_4 = sp.integrate(2*x,(x,0,3))
print(f"\nExercício 4 — integral definida: {integral_4}")

# EXERCÍCIO 5 — área entre y=x e y=x² em [0,1]
area_5 = sp.integrate(sp.Abs(x-x**2),(x,0,1))
print(f"\nExercício 5 — área entre as curvas: {area_5}")

## Síntese

Limites descrevem o comportamento de uma função perto de um ponto ou no infinito. Derivadas modelam taxas de variação instantânea e ajudam a analisar tangentes, crescimento, extremos e otimização. Integrais representam acumulação e permitem calcular áreas, deslocamentos, valores médios e volumes.

Ao resolver um problema computacionalmente, identifique a expressão e suas variáveis, verifique domínio e hipóteses, escolha a operação simbólica ou numérica adequada e interprete o resultado no contexto.